In [1]:
import pandas as pd
import numpy as np

# === Load data
real_data = pd.read_csv("law_real_with_model_predictions.csv")
synthetic_data = pd.read_csv("law_our_with_model_predictions.csv")

# === Parameters
sensitive_col = 'race'
privileged_value = 7
model_cols = [
    'pred_decision_tree',
    'pred_logistic_regression',
    'pred_random_forest',
    'pred_svm',
    'pred_xgboost'
]

# === Fairness Metric Function
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluation function
def evaluate_fairness_for_all(data, source_name):
    results = []
    for model in model_cols:
        metrics = compute_fairness(
            y_true=data['true_label'],
            y_pred=data[model],
            sensitive_attr=data[sensitive_col],
            privileged_value=privileged_value
        )
        result = {"Model": model, "Source": source_name, **metrics}
        results.append(result)
    return results

# === Evaluate both datasets
real_results = evaluate_fairness_for_all(real_data, 'Real')
synthetic_results = evaluate_fairness_for_all(synthetic_data, 'Synthetic')

# === Combine and format results
all_results = pd.DataFrame(real_results + synthetic_results)

# Format float columns
float_cols = ['DPD', 'DI', '∆TPR (EoO)', '∆FPR', '∆PPV', '∆Accuracy', 'EOD']
all_results[float_cols] = all_results[float_cols].astype(float).round(4)

# === Print in tabular format
print(all_results.to_string(index=False))

# === Save to CSV
all_results.to_csv("fairness_results_bar_pass.csv", index=False)


c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


                   Model    Source    DPD     DI  ∆TPR (EoO)    ∆FPR    ∆PPV  ∆Accuracy    EOD
      pred_decision_tree      Real 0.1515 0.8430      0.1349  0.2117  0.0690     0.1664 0.2117
pred_logistic_regression      Real 0.0559 0.9441      0.0335  0.2093  0.0664     0.0898 0.2093
      pred_random_forest      Real 0.0629 0.9369      0.0409  0.1986  0.0665     0.0955 0.1986
                pred_svm      Real 0.0000 1.0000      0.0000  0.0000  0.0871     0.0871 0.0000
            pred_xgboost      Real 0.0797 0.9200      0.0528  0.2620  0.0614     0.0993 0.2620
      pred_decision_tree Synthetic 0.0282 1.0565      0.0973 -0.0813  0.0685     0.0881 0.0973
pred_logistic_regression Synthetic 0.0000 1.0000      0.0973 -0.0221  0.0180     0.0599 0.0973
      pred_random_forest Synthetic 0.0328 1.0711      0.0573 -0.0473  0.0457     0.0542 0.0573
                pred_svm Synthetic 0.0662 0.8814      0.1081  0.0949 -0.0862     0.0050 0.1081
            pred_xgboost Synthetic 0.0221 1.0459  

In [2]:
import pandas as pd
import numpy as np

# === Load data
real_data = pd.read_csv("law_real_with_model_predictions.csv")
synthetic_data = pd.read_csv("law_cllm_with_model_predictions.csv")

# === Parameters
sensitive_col = 'race'
privileged_value = 7
model_cols = [
    'pred_decision_tree',
    'pred_logistic_regression',
    'pred_random_forest',
    'pred_svm',
    'pred_xgboost'
]

# === Fairness Metric Function
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluation function with printing
def evaluate_fairness_for_all(data, source_name):
    results = []
    for model in model_cols:
        metrics = compute_fairness(
            y_true=data['true_label'],
            y_pred=data[model],
            sensitive_attr=data[sensitive_col],
            privileged_value=privileged_value
        )
        result = {"Model": model, "Source": source_name, **metrics}
        results.append(result)
    return results

# === Evaluate datasets
real_results = evaluate_fairness_for_all(real_data, 'Real')
synthetic_results = evaluate_fairness_for_all(synthetic_data, 'Synthetic')

# === Combine and export
all_results = pd.DataFrame(real_results + synthetic_results)

# Format float columns
float_cols = ['DPD', 'DI', '∆TPR (EoO)', '∆FPR', '∆PPV', '∆Accuracy', 'EOD']
all_results[float_cols] = all_results[float_cols].astype(float).round(4)

# === Print in formatted table
print("\n=== Fairness Results on Real and CLLM-Synthetic Data ===")
print(all_results.to_string(index=False))

# === Save to CSV
all_results.to_csv("fairness_results_bar_passcllm.csv", index=False)



=== Fairness Results on Real and CLLM-Synthetic Data ===
                   Model    Source    DPD     DI  ∆TPR (EoO)    ∆FPR    ∆PPV  ∆Accuracy    EOD
      pred_decision_tree      Real 0.1515 0.8430      0.1349  0.2117  0.0690     0.1664 0.2117
pred_logistic_regression      Real 0.0559 0.9441      0.0335  0.2093  0.0664     0.0898 0.2093
      pred_random_forest      Real 0.0629 0.9369      0.0409  0.1986  0.0665     0.0955 0.1986
                pred_svm      Real 0.0000 1.0000      0.0000  0.0000  0.0871     0.0871 0.0000
            pred_xgboost      Real 0.0797 0.9200      0.0528  0.2620  0.0614     0.0993 0.2620
      pred_decision_tree Synthetic 0.0628 0.8937      0.0460 -0.0209  0.0401     0.0388 0.0460
pred_logistic_regression Synthetic 0.1078 0.8176      0.0690  0.0451 -0.0188     0.0163 0.0690
      pred_random_forest Synthetic 0.1078 0.8176      0.0575  0.0560 -0.0304     0.0051 0.0575
                pred_svm Synthetic 0.0286 0.9476      0.0316 -0.0648  0.0797     0.0495

In [4]:
import pandas as pd
import numpy as np

# === Load data
real_data = pd.read_csv("law_real_with_model_predictions.csv")
synthetic_data = pd.read_csv("law_decaf_with_model_predictions.csv")

# === Parameters
sensitive_col = 'race'
privileged_value = 7
model_cols = [
    'pred_decision_tree',
    'pred_logistic_regression',
    'pred_random_forest',
    'pred_svm',
    'pred_xgboost'
]

# === Fairness Metric Function
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluation function with printing
def evaluate_fairness_for_all(data, source_name):
    results = []
    for model in model_cols:
        metrics = compute_fairness(
            y_true=data['true_label'],
            y_pred=data[model],
            sensitive_attr=data[sensitive_col],
            privileged_value=privileged_value
        )
        result = {"Model": model, "Source": source_name, **metrics}
        results.append(result)
    return results

# === Evaluate datasets
real_results = evaluate_fairness_for_all(real_data, 'Real')
synthetic_results = evaluate_fairness_for_all(synthetic_data, 'Synthetic')

# === Combine results
all_results = pd.DataFrame(real_results + synthetic_results)

# Format float columns
float_cols = ['DPD', 'DI', '∆TPR (EoO)', '∆FPR', '∆PPV', '∆Accuracy', 'EOD']
all_results[float_cols] = all_results[float_cols].astype(float).round(4)

# === Print in tabular format
print("\n=== Fairness Results on Real and DECAF-Synthetic Data ===")
print(all_results.to_string(index=False))

# === Save to CSV
all_results.to_csv("fairness_results_bar_passcllm_decaf.csv", index=False)



=== Fairness Results on Real and DECAF-Synthetic Data ===
                   Model    Source    DPD     DI  ∆TPR (EoO)   ∆FPR    ∆PPV  ∆Accuracy    EOD
      pred_decision_tree      Real 0.1515 0.8430      0.1349 0.2117  0.0690     0.1664 0.2117
pred_logistic_regression      Real 0.0559 0.9441      0.0335 0.2093  0.0664     0.0898 0.2093
      pred_random_forest      Real 0.0629 0.9369      0.0409 0.1986  0.0665     0.0955 0.1986
                pred_svm      Real 0.0000 1.0000      0.0000 0.0000  0.0871     0.0871 0.0000
            pred_xgboost      Real 0.0797 0.9200      0.0528 0.2620  0.0614     0.0993 0.2620
      pred_decision_tree Synthetic 0.0299 1.0309     -0.0312 1.0000 -0.0432    -0.0719 1.0000
pred_logistic_regression Synthetic 0.0000 1.0000      0.0000 1.0000 -0.0419    -0.0419 1.0000
      pred_random_forest Synthetic 0.0000 1.0000      0.0000 1.0000 -0.0419    -0.0419 1.0000
                pred_svm Synthetic 0.0000 1.0000      0.0000 1.0000 -0.0419    -0.0419 1.0000
 